In [27]:
import sys

sys.path.insert(0, "/mmfs1/home/hardiksr/repository/hiatus/src")

import argparse
import json
import time
from pathlib import Path
import os
import wandb
import pandas as pd

from hiatus.config import metrics, metrics_dir

In [28]:
os.environ["WANDB_API_KEY"] = "c6ab8c3fa1999305c92ec52d8e537ddaf126bcc5"

In [29]:
# Write sampling config at the end of each run

def get_run_config(run_id: str) -> dict:
    config_filepath = ""
    with open(config_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [30]:
def initialize_wandb(run_id: str):
    # config = get_run_config(run_id=run_id)
    config = {
            "decoding_strategy": "default",
            "n": "1",
            "temperature": "0.55",
            "top_p": "0.9",
            "top_k": "50",
            "repetition_penalty": "1.1",
            "max_tokens": "800",
            "model_name": "inceptionai/jais-adapted-70b-chat"
        }

    init_args = {
        "project": "hiatus",
        "name": run_id,
        "entity": "zh-lab",
        "mode": "online",
        "config": config
    }

    return init_args

In [77]:
metrics_dir = "/gscratch/stf/hardiksr/hiatus/metrics/ar/saudinews"

In [78]:
metrics_path = {
    "prompt-remix-ar": {
        "sense_scores": [p.as_posix() for p in Path(metrics_dir).glob("*.jsonl")],
        # "privacy_consistency": [p.as_posix() for p in Path(metrics_dir, "ar").glob("*.csv")],
        "color": "#DF6800",
        "label": "PromptRemix-AR0",
        "hatch": False,
        "report": True
    }
}

In [79]:
def load_results(paths, sense_scores=True) -> pd.DataFrame:
    results = pd.DataFrame()
    for path in paths:
        if sense_scores:
            data = pd.read_json(path, lines=True)
        else:
            data = pd.read_csv(path)

        results = pd.concat([results, data])
    return results

In [80]:
def transform_data(metrics_path):
    df = pd.DataFrame()
    for model_name in metrics_path.keys():
        if metrics_path[model_name]["report"] == False:
            continue

        results = load_results(metrics_path[model_name]['sense_scores'])
        results = results.drop(columns="documentID", errors="ignore")
        
        # Apply scaling for each column
        for col in results.columns:
            if col in metrics:
                results[col] = results[col] * metrics[col]["scale"]

        results = results.mean(axis=0).to_frame(name="mean")
        results.index = [
                metrics[key]["title"] if key in metrics else key for key in results.index
            ]
        results = results.T

        df = pd.concat([df, results])
    return df

In [81]:
def log_wandb_table(run: wandb.Run, df: pd.DataFrame, init_args: dict):
    data = []
    columns = df.columns.tolist()
    mean_sense_scores = df.values.tolist()[0]

    for column, value in zip(columns, mean_sense_scores):
        data.append([column, value])

    table = wandb.Table(data=data, columns=["metric", "mean_sense_score"])
    plot_id = init_args['name']

    run.log({
        plot_id : wandb.plot.bar(
                    table, "metric", "mean_sense_score", title="Sense Scores"
                )
    })

In [83]:
def run_iter(metrics_path: dict):
    try:
        # run_id = metrics_filepath[metrics_filepath.index("promptremix"):].split(".")[0].replace("_", "-")
        run_id = "saudinews-promptremix-ar0"
        print(f"Executing process for run id: {run_id}")
        init_args = initialize_wandb(run_id=run_id)
        run = wandb.init(**init_args)
        df = transform_data(metrics_path=metrics_path)
        log_wandb_table(run=run, df=df, init_args=init_args)
        run.finish(0)
    except Exception as e:
        print(e)

In [84]:
run_iter(metrics_path=metrics_path)

Executing process for run id: saudinews-promptremix-ar0


In [ ]:
# api = wandb.Api()
# runs = api.runs(path=f"zh-lab/hiatus")

In [ ]:
# for run in runs:
#     print(run.name, run.id)

promptremix-TA3-ar2 16pbc1mo
promptremix-TA3-ar3 7cm4qdoi
promptremix-TA3-ar0 j94lz66o
promptremix-TA3-ar3-1 vbkprtao
promptremix-TA3-ar4 b4qcr899
promptremix-TA3-ar5 cxwf4deo
promptremix-TA3-ar6 ww52dbrk
promptremix-TA3-ar6-1 5jd37yz4


In [ ]:
# run = api.run(f"zh-lab/hiatus/j94lz66o")
# run.config["model_name"] = "inceptionai/jais-adapted-7b-chat"
# run.update()

In [ ]:
# for run in runs:
#     run = api.run(f"zh-lab/hiatus/{run.id}")
#     run.config["model_name"] = ""
#     run.update()